In [226]:
from pathlib import Path
import urllib.request
import torch

In [227]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

device

'cuda'

In [228]:
def download_shakespeare_text():
    path = Path("datasets/shakespeare/shakespeare.txt")
    if not path.is_file():
        path.parent.mkdir(parents=True, exist_ok=True)
        url = "https://homl.info/shakespeare"
        urllib.request.urlretrieve(url, path)
    return path.read_text()

In [229]:
%pip install torchmetrics 

Note: you may need to restart the kernel to use updated packages.


In [230]:
import torchmetrics

def evaluate_tm(model, valid_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in valid_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()

def train(model, optimizer, criterion, metric, train_loader, valid_loader, n_epochs, patience = 2, factor = 0.5, epoch_callback = None):
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode = "max", patience=patience, factor = factor)
    history = {"train_losses" : [], "train_metrics" : [], "valid_metrics" : []}
    for epoch in range(n_epochs):
        total_loss = 0
        metric.reset()
        model.train()
        if epoch_callback is not None:
            epoch_callback(model, epoch)
        for index, (X_batch, y_batch) in enumerate(train_loader):
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
            train_metric = metric.compute().item()
            print(f"\rBatch {index + 1} / {len(train_loader)}", end="")
            print(f", loss={total_loss/(index+1):.4f}", end="")
            print(f", {train_metric=:.2%}", end="")
        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(train_metric)
        val_metric = evaluate_tm(model, valid_loader, metric).item()
        history["valid_metrics"].append(val_metric)
        scheduler.step(val_metric)
        print(f"\rEpoch {epoch + 1}/{n_epochs},                      "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.2%}, "
              f"valid metric: {history['valid_metrics'][-1]:.2%}")
    return history




In [231]:
import gc

def del_vars(variable_list = []):
    for name in variable_list:
        try:
            del globals()[name]
        except:
            pass
    gc.collect()

    if device == "cuda":
        torch.cuda.empty_cache()

In [232]:
shakespeare_text = download_shakespeare_text()

In [233]:
len(shakespeare_text)

1115394

In [234]:
print(shakespeare_text[:80])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.


In [235]:
vocab = sorted(set(shakespeare_text.lower()))

In [236]:
"".join(vocab)

"\n !$&',-.3:;?abcdefghijklmnopqrstuvwxyz"

In [237]:
char_to_id = {char : index for index, char in enumerate(vocab)}
id_to_char = {index : char for index, char in enumerate(vocab)}



In [238]:
char_to_id["a"]

13

In [239]:
char_to_id["!"]

2

In [240]:
id_to_char[13]

'a'

In [241]:
import torch

def text_to_tensor(text):
    return torch.tensor([char_to_id[char] for char in text.lower()])

In [242]:
def tensor_to_text(tensor):
    return "".join([id_to_char[id.item()] for id in tensor])

In [243]:
tensor_to_text(text_to_tensor(shakespeare_text[:80]))

'first citizen:\nbefore we proceed any further, hear me speak.\n\nall:\nspeak, speak.'

In [244]:
from torch.utils.data import DataLoader, Dataset

class CharDataset(Dataset):
    def __init__(self, text, window_length):
        # super().__init__()
        self.encoded_text = text_to_tensor(text)
        self.window_length = window_length
    def __len__(self):
        return len(self.encoded_text) - self.window_length
    def __getitem__(self, index):
        if index >= len(self):
            raise IndexError("Index Out of Bounds")
        end = index + self.window_length
        window = self.encoded_text[index : end]
        target = self.encoded_text[index  + 1 : end + 1]
        return window, target
    

In [245]:
window_length = 50
batch_size = 512
train_dataset = CharDataset(shakespeare_text[:1000000], window_length)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_dataset = CharDataset(shakespeare_text[1000000:1060000], window_length)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size)
test_dataset = CharDataset(shakespeare_text[1060000:], window_length)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

In [246]:
import torch.nn as nn

In [247]:
class ShakespeareModel(nn.Module):
    def __init__(self, vocab_size, hidden_dim = 128, num_layers = 2, dropout = 0.1, embed_dim = 10):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embedding_dim= embed_dim)
        self.GRU = nn.GRU(input_size=embed_dim, hidden_size=hidden_dim, dropout=dropout, batch_first=True, num_layers=num_layers)
        self.output = nn.Linear(hidden_dim, out_features=vocab_size)
    def forward(self, X):
        embeddings = self.embed(X)
        H, _states = self.GRU(embeddings)
        return self.output(H).permute(0,2,1)

In [248]:
torch.manual_seed(seed = 42)
model = ShakespeareModel(vocab_size=len(vocab)).to(device)

In [249]:
n_epochs = 20
xentropy = nn.CrossEntropyLoss()
optimizer = torch.optim.NAdam(model.parameters())
accuracy = torchmetrics.Accuracy(task="multiclass",
                                 num_classes=len(vocab)).to(device)

history = train(model, optimizer, xentropy, accuracy, train_loader, valid_loader,
                n_epochs)

Epoch 1/20,                      train loss: 1.6042, train metric: 51.27%, valid metric: 51.71%
Epoch 2/20,                      train loss: 1.3843, train metric: 56.71%, valid metric: 53.09%
Epoch 3/20,                      train loss: 1.3548, train metric: 57.44%, valid metric: 53.38%
Epoch 4/20,                      train loss: 1.3406, train metric: 57.80%, valid metric: 53.94%
Epoch 5/20,                      train loss: 1.3322, train metric: 58.02%, valid metric: 53.62%
Epoch 6/20,                      train loss: 1.3267, train metric: 58.16%, valid metric: 54.28%
Epoch 7/20,                      train loss: 1.3226, train metric: 58.27%, valid metric: 53.95%
Epoch 8/20,                      train loss: 1.3196, train metric: 58.34%, valid metric: 54.08%
Epoch 9/20,                      train loss: 1.3170, train metric: 58.39%, valid metric: 54.26%
Epoch 10/20,                      train loss: 1.3080, train metric: 58.66%, valid metric: 54.94%
Epoch 11/20,                      train

In [250]:
model.eval()
text = "To be or not to b"
tokeninzed_text = text_to_tensor(text).unsqueeze(dim = 0)

In [251]:
model.eval()
with torch.no_grad():
    y_logits = model(tokeninzed_text.to(device))
    index = y_logits[0, :, -1].argmax().item()
    predicted_char = id_to_char[index]

In [252]:
predicted_char

'e'

In [253]:
torch.manual_seed(seed = 42)
probs = torch.tensor([[0.5,0.4,0.1]])
samples = torch.multinomial(probs, num_samples=8, replacement=True)
samples

tensor([[0, 0, 0, 0, 1, 0, 2, 2]])

In [254]:
import torch.nn.functional as F

def next_char(model, text, temperature = 1):
    encoded_text = text_to_tensor(text).unsqueeze(dim = 0).to(device)
    model.eval()
    with torch.no_grad():
        y_logits = model(encoded_text)
        y_softmax = F.softmax(y_logits[0, :, -1] / temperature, dim = -1)
        next_char_index = torch.multinomial(y_softmax, num_samples=1).item()
    return id_to_char[next_char_index]

In [255]:
def extend_text(model, text, temperature, n_chars = 500):
    for _ in range(n_chars):
        text += next_char(model, text, temperature)
    return text

In [256]:
print(extend_text(model, "hey are dissolved: hang 'em!", temperature=0.4))

hey are dissolved: hang 'em!

second murderer:
and therefore is the people.

gloucester:
as i may be some intent of the envious treason.

menenius:
he was not my heart.

coriolanus:
how now! why, then, i shall find the seat of my father's death.

menenius:
the rest in the friends and the love
that we have in the seat of such a strange usurp'd to the sanctuary deliver'd in the shadow
the capulets and weeds in the breath to the duke of gloucester:
the duke of norfolk, and with the sight of the beaten and friends,
and they sh


In [257]:
%pip install datasets

Note: you may need to restart the kernel to use updated packages.


In [258]:
from datasets import load_dataset

In [259]:
# 1. Upgrade the libraries without leaving the notebook
!pip install --upgrade datasets huggingface_hub

# 2. Force Python to reload the modules in the current session
import importlib
import datasets
import huggingface_hub

importlib.reload(huggingface_hub)
importlib.reload(datasets)

print("Hugging Face libraries reloaded successfully without kernel restart!")

Hugging Face libraries reloaded successfully without kernel restart!


In [260]:
imdb_datasets = load_dataset("stanfordnlp/imdb")
split = imdb_datasets["train"].train_test_split(train_size=0.8, seed = 42)
imdb_train_set, imdb_valid_set = split["train"], split["test"]
imdb_test_set = imdb_datasets["test"]

In [261]:
imdb_train_set[1]["text"]

"'The Rookie' was a wonderful movie about the second chances life holds for us and also puts an emotional thought over the audience, making them realize that your dreams can come true. If you loved 'Remember the Titans', 'The Rookie' is the movie for you!! It's the feel good movie of the year and it is the perfect movie for all ages. 'The Rookie' hits a major home run!"

In [262]:
imdb_train_set[1]["label"]

1

In [263]:
imdb_train_set[16]["text"]

"Lillian Hellman's play, adapted by Dashiell Hammett with help from Hellman, becomes a curious project to come out of gritty Warner Bros. Paul Lukas, reprising his Broadway role and winning the Best Actor Oscar, plays an anti-Nazi German underground leader fighting the Fascists, dragging his American wife and three children all over Europe before finding refuge in the States (via the Mexico border). They settle in Washington with the wife's wealthy mother and brother, though a boarder residing in the manor is immediately suspicious of the newcomers and spends an awful lot of time down at the German Embassy playing poker. It seems to take forever for this drama to find its focus, and when we realize what the heart of the material is (the wise, honest, direct refugees teaching the clueless, head-in-the-sand Americans how the world has suddenly changed), it seems a little patronizing--the viewer is quite literally put in the relatives' place, being lectured to. Lukas has several speeches 

In [264]:
%pip install tokenizers

Note: you may need to restart the kernel to use updated packages.


In [265]:
import tokenizers

In [266]:
bpe_model = tokenizers.models.BPE(unk_token="<unk>")
bpe_tokenizer = tokenizers.Tokenizer(bpe_model)
bpe_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()
special_tokens = ["<pad>", "<unk>"]
bpe_trainer = tokenizers.trainers.BpeTrainer(vocab_size=1000, special_tokens = special_tokens)
reviews = [review["text"].lower() for review in imdb_train_set]
bpe_tokenizer.train_from_iterator(reviews, bpe_trainer)

In [267]:
some_review = "what an awesome movie! 😊"
bpe_encoded = bpe_tokenizer.encode(some_review)
bpe_encoded

Encoding(num_tokens=8, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])

In [268]:
bpe_encoded.ids

[303, 139, 373, 149, 240, 211, 4, 1]

In [269]:
bpe_encoded.tokens

['what', 'an', 'aw', 'es', 'ome', 'movie', '!', '<unk>']

In [270]:
bpe_token_ids = bpe_encoded.ids

In [271]:
bpe_encoded.offsets

[(0, 4), (5, 7), (8, 10), (10, 12), (12, 15), (16, 21), (21, 22), (23, 24)]

In [272]:
bpe_tokenizer.decode(bpe_token_ids)

'what an aw es ome movie !'

In [273]:
bpe_tokenizer.enable_padding(pad_id=0, pad_token="<pad>")
bpe_tokenizer.enable_truncation(max_length=500)

In [274]:
bpe_encodings = bpe_tokenizer.encode_batch(reviews[:3])

In [275]:
bpe_batch_ids = torch.tensor([encoding.ids for encoding in bpe_encodings])


In [276]:
bpe_batch_ids

tensor([[159, 402, 176, 246,  61, 782, 156, 737, 252,  42, 239,  51, 154, 460,
         917,  17, 272, 156, 737, 576, 215, 976, 275,  42, 199,  44, 554,  42,
         192, 585,  57, 160, 259, 170, 157, 143, 138, 159, 402,  11, 589, 152,
           5, 819, 168, 230,   5, 521, 924, 981, 962, 250,  61,  10,  60, 426,
         526, 959,  60, 138, 199, 150, 319,  15, 363, 141, 957, 694,  47, 696,
          61, 875, 138, 960, 337, 414, 140, 157, 385, 174, 433, 161, 221, 145,
         213,  17, 549,  15, 151,  10,  60,  55, 416, 146, 407, 144, 182, 303,
         151, 141,  17, 138, 547, 538, 528, 768,  54, 335,  42, 203,  44, 270,
          46, 153, 876, 141, 919, 233, 522, 172, 141, 719, 162, 807, 279,  17,
         138,  45,  66,  55, 188, 989, 156, 378, 698, 301, 296, 689, 212, 558,
         926, 148,  17,  44, 270,  46, 141,  47, 279, 302, 171, 152, 787,  15,
         153, 522, 172, 766, 205, 156, 234, 677, 161, 139, 513, 146, 370, 251,
         219, 162, 197, 162, 166,  50, 265,  47, 266

In [277]:
attention = torch.tensor([encodings.attention_mask for encodings in bpe_encodings])

In [278]:
attention

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 

In [279]:
attention.sum(dim = -1)

tensor([281, 114, 285])

In [280]:
%pip install transformers

Note: you may need to restart the kernel to use updated packages.


In [281]:
import transformers

In [282]:
gpt2_tokenizer = transformers.AutoTokenizer.from_pretrained("gpt2")

In [283]:
gpt2_encoding = gpt2_tokenizer(reviews[:3], max_length = 500, truncation=True)

In [284]:
gpt2_encoding.keys()

KeysView({'input_ids': [[14247, 35030, 1690, 423, 257, 1688, 8046, 13, 484, 1690, 1282, 503, 2045, 588, 257, 2646, 4676, 373, 2391, 4624, 319, 262, 3800, 357, 10508, 355, 366, 3847, 2802, 11074, 9785, 1681, 46390, 316, 338, 4571, 7622, 262, 2646, 6776, 11, 543, 318, 2592, 2408, 1201, 262, 4286, 4438, 683, 645, 1103, 4427, 13, 991, 11, 340, 338, 3621, 284, 804, 379, 329, 644, 340, 318, 13, 262, 16585, 1022, 285, 40302, 269, 5718, 290, 33826, 8803, 302, 44655, 318, 2407, 10457, 13, 262, 17262, 286, 511, 2776, 389, 6452, 13, 269, 5718, 318, 9623, 355, 1464, 11, 290, 302, 44655, 3011, 530, 286, 465, 1178, 8395, 284, 1107, 719, 29847, 1671, 1220, 6927, 1671, 11037, 72, 22127, 326, 1312, 1053, 1239, 1775, 4173, 64, 443, 7114, 338, 711, 11, 475, 1312, 3285, 326, 474, 323, 1803, 261, 477, 268, 338, 16711, 318, 17074, 13, 262, 4226, 318, 8131, 47370, 11, 290, 7622, 345, 25260, 13, 366, 22595, 46670, 1, 318, 281, 36005, 17774, 2646, 11, 290, 318, 7151, 329, 3016, 477, 3296, 286, 3800, 290, 3159,

In [285]:
gpt2_token_ids = gpt2_encoding["input_ids"][0][:10]

In [286]:
gpt2_tokenizer.decode(gpt2_token_ids)

'stage adaptations often have a major fault. they often'

In [287]:
bert_tokenizer = transformers.AutoTokenizer.from_pretrained("bert-base-uncased")

In [288]:
bert_encodings = bert_tokenizer(reviews[:3], padding = True, truncation = True, max_length = 500, return_tensors = "pt")

In [289]:
bert_encodings["input_ids"]

tensor([[  101,  2754, 17241,  2411,  2031,  1037,  2350,  6346,  1012,  2027,
          2411,  2272,  2041,  2559,  2066,  1037,  2143,  4950,  2001,  3432,
          2872,  2006,  1996,  2754,  1006,  2107,  2004,  1000,  2305,  2388,
          1000,  1007,  1012, 11430, 11320, 11368,  1005,  1055,  3257,  7906,
          1996,  2143,  4142,  1010,  2029,  2003,  2926,  3697,  2144,  1996,
          3861,  3253,  2032,  2053,  2613,  4119,  1012,  2145,  1010,  2009,
          1005,  1055,  3835,  2000,  2298,  2012,  2005,  2054,  2009,  2003,
          1012,  1996,  6370,  2090,  2745, 19881,  1998,  5696, 20726,  2003,
          3243,  8235,  1012,  1996, 10949,  1997,  2037,  3276,  2024, 11341,
          1012, 19881,  2003, 10392,  2004,  2467,  1010,  1998, 20726,  4152,
          2028,  1997,  2010,  2261,  9592,  2000,  2428,  2552,  1012,  1026,
          7987,  1013,  1028,  1026,  7987,  1013,  1028,  1045, 18766,  2008,
          1045,  1005,  2310,  2196,  2464, 11209, 2

In [290]:
bert_token_ids = bert_encodings["input_ids"][0][:10]

In [291]:
bert_tokenizer.decode(bert_token_ids)

'[CLS] stage adaptations often have a major fault. they'

In [292]:
bert_encodings["attention_mask"]

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 

In [293]:
hf_tokenizer = transformers.PreTrainedTokenizerFast(tokenizer_object = bpe_tokenizer)
hf_encodings = hf_tokenizer(reviews[:3], padding = True, truncation=True, max_length=500, return_tensors="pt")

In [294]:
def collate_fn(batch, tokenizer = bert_tokenizer):
    reviews = [review["text"] for review in batch]
    labels = [[review["label"]] for review in batch]
    encoded_reviews = tokenizer(reviews, truncation = True, max_length = 200, padding = True, return_tensors = "pt")
    labels = torch.tensor(labels, dtype = torch.float32)
    return encoded_reviews, labels

In [295]:
batch_size = 256
imdb_train_loader = DataLoader(imdb_train_set, collate_fn=collate_fn, batch_size = 256, shuffle=True)

In [296]:
imdb_valid_loader = DataLoader(imdb_valid_set, batch_size=batch_size,
                               collate_fn=collate_fn)
imdb_test_loader = DataLoader(imdb_test_set, batch_size=batch_size,
                              collate_fn=collate_fn)

In [297]:
class SentimentAnalysisModel(nn.Module):
    def __init__(self, vocab_size , hidden_size, embed_dim = 128, n_layers = 2, dropout = 0.2, pad_id = 0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.GRU = nn.GRU(embed_dim, hidden_size, n_layers = 2, dropout=dropout, batch_first=True)
        self.output = nn.Linear(hidden_size, 1)
    def forward(self, X):
        embeddings = self.embedding(X["input_ids"])
        _outputs, hidden_state = self.GRU(embeddings)
        return self.output(_outputs[:, -1])

In [298]:
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

sequences = torch.tensor([[1,2,0,0],[5,6,7,8]])
padded = pack_padded_sequence(sequences, enforce_sorted=False, lengths = (2,4), batch_first = True)

In [299]:
padded

PackedSequence(data=tensor([5, 1, 6, 2, 7, 8]), batch_sizes=tensor([2, 2, 1, 1]), sorted_indices=tensor([1, 0]), unsorted_indices=tensor([1, 0]))

In [300]:
sequence, length = pad_packed_sequence(padded, batch_first=True)

In [301]:
sequence, length

(tensor([[1, 2, 0, 0],
         [5, 6, 7, 8]]),
 tensor([2, 4]))

In [305]:
class SentimentAnalysisModelPackedSeq(nn.Module):
    def __init__(self, vocab_size , hidden_size = 64, embed_dim = 128, num_layers = 2, dropout = 0.2, pad_id = 0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.GRU = nn.GRU(embed_dim, hidden_size, num_layers = 2, dropout=dropout, batch_first=True)
        self.output = nn.Linear(hidden_size, 1)
    def forward(self, X):
        embeddings = self.embedding(X["input_ids"])
        lengths = X["attention_mask"].sum(dim = 1)
        packed_embeddings = pack_padded_sequence(embeddings, enforce_sorted=False, batch_first=True, lengths=lengths.cpu())
        _outputs, hidden_state = self.GRU(packed_embeddings)
        return self.output(hidden_state[-1])

In [306]:
torch.manual_seed(42)

vocab_size = bert_tokenizer.vocab_size
imdb_model_ps = SentimentAnalysisModelPackedSeq(vocab_size).to(device)

n_epochs = 10
xentropy = nn.BCEWithLogitsLoss()
optimizer = torch.optim.NAdam(imdb_model_ps.parameters())
accuracy = torchmetrics.Accuracy(task="binary").to(device)

history = train(imdb_model_ps, optimizer, xentropy, accuracy,
                imdb_train_loader, imdb_valid_loader, n_epochs)

Epoch 1/10,                      train loss: 0.6748, train metric: 58.00%, valid metric: 62.02%
Epoch 2/10,                      train loss: 0.5874, train metric: 69.85%, valid metric: 67.32%
Epoch 3/10,                      train loss: 0.4933, train metric: 76.38%, valid metric: 79.08%
Epoch 4/10,                      train loss: 0.3446, train metric: 85.30%, valid metric: 83.60%
Epoch 5/10,                      train loss: 0.2609, train metric: 89.23%, valid metric: 82.20%
Epoch 6/10,                      train loss: 0.1867, train metric: 93.09%, valid metric: 85.08%
Epoch 7/10,                      train loss: 0.1345, train metric: 95.26%, valid metric: 84.96%
Epoch 8/10,                      train loss: 0.0991, train metric: 97.00%, valid metric: 85.44%
Epoch 9/10,                      train loss: 0.0515, train metric: 98.70%, valid metric: 84.32%
Epoch 10/10,                      train loss: 0.0757, train metric: 98.04%, valid metric: 83.34%


In [ ]:
# Out.clear()  # clear Jupyter's `Out` variable which saves all the cell outputs
# del_vars(["albert_token_ids", "attention_mask", "bpe_batch_ids", "encoded_text",
#           "lengths", "optimizer", "padded", "probs", "samples", "sequences",
#           "x", "xentropy", "y", "Y_logits"])

In [312]:
class SentimentAnalysisModelBidi(nn.Module):
    def __init__(self, vocab_size, n_layers=2, embed_dim=128,
                 hidden_dim=64, pad_id=0, dropout=0.2):
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.GRU = nn.GRU(embed_dim, hidden_dim, dropout=dropout, num_layers=n_layers, batch_first=True, bidirectional=True)
        self.output = nn.Linear(2 * hidden_dim, 1)
    def forward(self, X):
        embeddings = self.embedding(X["input_ids"])
        lengths = X["attention_mask"].sum(dim = 1)
        packed_sequence = pack_padded_sequence(embeddings, lengths=lengths.cpu(), enforce_sorted=False, batch_first=True)
        _outputs, hidden_states = nn.GRU(packed_sequence)
        n_dims = self.output.in_features
        top_states = hidden_states[-2:].permute(1,0,2).reshape(-1,n_dims)
        return self.output(top_states)

In [308]:
bert_model = transformers.AutoModel.from_pretrained("bert-base-uncased")
bert_model.embeddings.word_embeddings

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding(30522, 768, padding_idx=0)

In [313]:
class SentimentAnalysisModelPreEmbeds(nn.Module):
    def __init__(self, pretrained_embeddings, hidden_size = 64, num_layers = 2, dropout = 0.2):
        super().__init__()
        weights = pretrained_embeddings.weight.data
        self.embeddings = nn.Embedding.from_pretrained(weights, freeze=True)
        self.embed_dim = weights.shape[-1]
        self.GRU = nn.GRU(self.embed_dim, hidden_size, num_layers=num_layers, dropout=dropout, batch_first=True, bidirectional=True)
        self.output = nn.Linear(hidden_size * 2, 1)
    def forward(self, X):
        embeddings = self.embeddings(X["input_ids"])
        lengths = X["attention_mask"].sum(dim = 1)
        packed_sequence = pack_padded_sequence(embeddings, lengths=lengths.cpu(), enforce_sorted=False, batch_first=True)
        _outputs, hidden_states = self.GRU(packed_sequence)
        n_dims = self.output.in_features
        top_states = hidden_states[-2:].permute(1,0,2).reshape(-1,n_dims)
        return self.output(top_states)

In [314]:
torch.manual_seed(42)

imdb_model_bert_embeds = SentimentAnalysisModelPreEmbeds(
    bert_model.embeddings.word_embeddings).to(device)

n_epochs = 10
xentropy = nn.BCEWithLogitsLoss()
optimizer = torch.optim.NAdam(imdb_model_bert_embeds.parameters())
accuracy = torchmetrics.Accuracy(task="binary").to(device)

history = train(imdb_model_bert_embeds, optimizer, xentropy, accuracy,
                imdb_train_loader, imdb_valid_loader, n_epochs)

Epoch 1/10,                      train loss: 0.6892, train metric: 55.41%, valid metric: 63.12%
Epoch 2/10,                      train loss: 0.6379, train metric: 64.40%, valid metric: 66.78%
Epoch 3/10,                      train loss: 0.5519, train metric: 72.54%, valid metric: 75.72%
Epoch 4/10,                      train loss: 0.4697, train metric: 77.72%, valid metric: 73.72%
Epoch 5/10,                      train loss: 0.4187, train metric: 80.53%, valid metric: 80.12%
Epoch 6/10,                      train loss: 0.3681, train metric: 83.13%, valid metric: 83.92%
Epoch 7/10,                      train loss: 0.3316, train metric: 85.43%, valid metric: 80.56%
Epoch 8/10,                      train loss: 0.3090, train metric: 86.51%, valid metric: 79.66%
Epoch 9/10,                      train loss: 0.2879, train metric: 87.70%, valid metric: 60.90%
Epoch 10/10,                      train loss: 0.2489, train metric: 89.62%, valid metric: 81.26%


In [315]:
bert_encodings = bert_tokenizer(reviews[:3], padding = True, max_length = 200, truncation = True, return_tensors = "pt")
bert_output = bert_model(**bert_encodings)

In [316]:
bert_output.last_hidden_state.shape

torch.Size([3, 200, 768])

In [317]:
class SentimentAnalysisModelBert(nn.Module):
    def __init__(self, n_layers = 2, n_hidden = 64, dropout = 0.2):
        super().__init__()
        self.bert = transformers.AutoModel.from_pretrained("bert-base-uncased")
        self.embed_dim = self.bert.config.hidden_size
        self.GRU = nn.GRU(self.embed_dim, n_hidden, num_layers=n_layers, dropout = dropout, batch_first = True)
        self.output = nn.Linear(n_hidden, 1)
    def forward(self, encodings):
        embeddings = self.bert(**encodings).last_hidden_state
        lengths = encodings["attention_mask"].sum(dim = 1)
        packed_sequence = pack_padded_sequence(embeddings, lengths = lengths.cpu(), enforce_sorted=False, batch_first=True)
        _outputs, hidden_states = self.GRU(packed_sequence)
        return self.output(hidden_states[-1])


In [318]:
torch.manual_seed(42)

imdb_model_bert = SentimentAnalysisModelBert().to(device)
imdb_model_bert.bert.requires_grad_(False)

n_epochs = 4
xentropy = nn.BCEWithLogitsLoss()
optimizer = torch.optim.NAdam(imdb_model_bert.parameters())
accuracy = torchmetrics.Accuracy(task="binary").to(device)

history = train(imdb_model_bert, optimizer, xentropy, accuracy,
                imdb_train_loader, imdb_valid_loader, n_epochs)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/4,                      train loss: 0.4872, train metric: 75.66%, valid metric: 87.20%
Epoch 2/4,                      train loss: 0.3014, train metric: 87.16%, valid metric: 88.50%
Epoch 3/4,                      train loss: 0.2718, train metric: 88.62%, valid metric: 86.96%
Epoch 4/4,                      train loss: 0.2552, train metric: 89.40%, valid metric: 86.24%


In [ ]:
class SentimentAnalysisModelBert2(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert_model = transformers.AutoModel.from_pretrained("bert-base-uncased")
        self.output = nn.Linear(self.bert_model.config.hidden_size, 1)
    def forward(self, encodings):
        bert_output = self.bert_model(**encodings)
        return self.output(bert_output.last_hidden_state[:,0,:])

In [320]:
class SentimentAnalysisModelBert3(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert_model = transformers.AutoModel.from_pretrained("bert-base-uncased")
        self.output = nn.Linear(self.bert_model.config.hidden_size, 1)
    def forward(self, encodings):
        bert_output = self.bert_model(**encodings)
        return self.output(bert_output.pooler_output)

In [ ]:
from transformers import BertForSequenceClassification

torch.manual_seed(seed = 42)
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels = 2, dtype = torch.float16).to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [322]:
encoding = bert_tokenizer(["That was a Great Movie!"])
with torch.no_grad():
    output = model(input_ids = torch.tensor(encoding["input_ids"], device = device), attention_mask = torch.tensor(encoding["attention_mask"], device = device))
    

In [324]:
output.logits

tensor([[ 0.1406, -0.2407]], device='cuda:0', dtype=torch.float16)

In [327]:
import torch.nn.functional as F


In [330]:
F.softmax(output.logits, dim = -1)

tensor([[0.5942, 0.4058]], device='cuda:0', dtype=torch.float16)

In [333]:
model.config

BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "dtype": "float16",
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "tie_word_embeddings": true,
  "transformers_version": "5.11.0",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

In [334]:
def tokenize_batch(batch):
    return bert_tokenizer(batch["text"], truncation = True, max_length = 100)

tok_imdb_train_set = imdb_train_set.map(tokenize_batch, batched=True)
tok_imdb_valid_set = imdb_valid_set.map(tokenize_batch, batched = True)
tok_imdb_test_set = imdb_test_set.map(tokenize_batch, batched = True)

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

In [335]:
def compute_accuracy(pred):
    return {"accuracy" : (pred.label_ids == pred.predictions.argmax(dim = -1)).mean()}

In [350]:
# 1. Double-check it actually installed in your environment
%pip install -q accelerate

# 2. Reach into Hugging Face's backend checking module
import transformers.utils.import_utils as hf_utils

# 3. Overwrite the function that is blocking you
hf_utils.is_accelerate_available = lambda: True

# 4. Overwrite the backend validator function so it completely skips the check
def dummy_requires_backends(obj, backends):
    return True

transformers.utils.import_utils.requires_backends = dummy_requires_backends

Note: you may need to restart the kernel to use updated packages.


In [354]:
# 1. Reach directly into the executing training_args module namespace
import transformers.training_args as ta

# 2. Obliterate the specific check function inside that file
ta.requires_backends = lambda obj, backends: True

# 3. Patch the inner import utilities inside that file just in case
ta.is_accelerate_available = lambda: True

# 4. Completely bypass the internal .device property setup that triggers the error 
# by hardcoding it to return your active device (e.g., "cuda" or "cpu")
import torch
current_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ta.TrainingArguments.device = property(lambda self: current_device)
ta.TrainingArguments._setup_devices = property(lambda self: current_device)

In [362]:
from transformers import TrainingArguments

train_args = TrainingArguments(
    output_dir = "my_imdb_model", num_train_epochs=2, per_device_train_batch_size=128, per_device_eval_batch_size=128, 
    eval_strategy="epoch", logging_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True, metric_for_best_model="accuracy", report_to="none"
)
train_args.accelerator_config.gradient_accumulation_kwargs = {}

In [356]:
import transformers.training_args as ta

# 1. Grab the actual AcceleratorConfig class from where it lives now
try:
    from accelerate import AcceleratorConfig
except ImportError:
    # Fallback dummy class if your python paths are being extra stubborn
    class AcceleratorConfig:
        def __init__(self, *args, **kwargs): pass
        def to_dict(self): return {}

# 2. Inject it straight into the training_args module namespace
ta.AcceleratorConfig = AcceleratorConfig

In [358]:
import transformers.training_args as ta

# Define a super-robust dummy class that dynamically fakes any property asked of it
class DynamicDummyConfig:
    def __init__(self, *args, **kwargs):
        pass
    def to_dict(self):
        return {}
    # If the code asks for .split_batches or anything else, just return False
    def __getattr__(self, name):
        return False

# Inject it back into the namespace
ta.AcceleratorConfig = DynamicDummyConfig

In [364]:
%pip install --upgrade "transformers>=4.40.0" "accelerate>=0.30.0"

   ---------------------------------------- 0.0/11.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.2 MB ? eta -:--:--
   - -------------------------------------- 0.5/11.2 MB 1.3 MB/s eta 0:00:08
   - -------------------------------------- 0.5/11.2 MB 1.3 MB/s eta 0:00:08
   -- ------------------------------------- 0.8/11.2 MB 1.2 MB/s eta 0:00:09
   --- ------------------------------------ 1.0/11.2 MB 1.3 MB/s eta 0:00:08
   ---- ----------------------------------- 1.3/11.2 MB 1.2 MB/s eta 0:00:08
   ------ --------------------------------- 1.8/11.2 MB 1.3 MB/s eta 0:00:08
   ------- -------------------------------- 2.1/11.2 MB 1.3 MB/s eta 0:00:08
   -------- ------------------------------- 2.4/11.2 MB 1.3 MB/s eta 0:00:07
   --------- ------------------------------ 2.6/11.2 MB 1.3 MB/s eta 0:00:07
   ---------- ----------------------------- 2.9/11.2 MB 1.3 MB/s eta 0:00:07
   ----------- ---------------------------- 3.1/11.2 MB 1.3 MB/s eta 0:00:07
   ----------

In [367]:
# 1. Import the missing plugin from accelerate
import accelerate
from accelerate.utils import GradientAccumulationPlugin

# 2. Inject it into the transformers trainer module dynamically
import transformers.trainer
transformers.trainer.GradientAccumulationPlugin = GradientAccumulationPlugin

# 3. Apply your previous workaround just in case
train_args.accelerator_config.gradient_accumulation_kwargs = {}

# 4. Now run your Trainer safely
from transformers import Trainer, DataCollatorWithPadding

trainer = Trainer(
    model=model, 
    args=train_args, 
    train_dataset=tok_imdb_train_set, 
    eval_dataset=tok_imdb_valid_set, 
    compute_metrics=compute_accuracy, 
    data_collator=DataCollatorWithPadding(bert_tokenizer)
)

train_output = trainer.train()

NameError: name 'DataLoaderConfiguration' is not defined

In [365]:
from transformers import Trainer, DataCollatorWithPadding

trainer = Trainer(model, train_args, train_dataset=tok_imdb_train_set, eval_dataset=tok_imdb_valid_set, compute_metrics=compute_accuracy, data_collator=DataCollatorWithPadding(bert_tokenizer))


NameError: name 'GradientAccumulationPlugin' is not defined

In [370]:
from transformers import pipeline
model_name = "distilbert-base-uncased-finetuned-sst-2-english"
classifier_imdb = pipeline("sentiment-analysis", model = model_name, truncation = True, max_lenth = 512)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [371]:
classifier_imdb(train_reviews[:10])

[{'label': 'POSITIVE', 'score': 0.9996108412742615},
 {'label': 'POSITIVE', 'score': 0.9998623132705688},
 {'label': 'NEGATIVE', 'score': 0.9943684935569763},
 {'label': 'POSITIVE', 'score': 0.997913658618927},
 {'label': 'POSITIVE', 'score': 0.999544084072113},
 {'label': 'NEGATIVE', 'score': 0.9845332503318787},
 {'label': 'POSITIVE', 'score': 0.9859278202056885},
 {'label': 'POSITIVE', 'score': 0.9993758797645569},
 {'label': 'POSITIVE', 'score': 0.9978922009468079},
 {'label': 'NEGATIVE', 'score': 0.9997020363807678}]

In [372]:
model_name = "huggingface/distilbert-base-uncased-finetuned-mnli"
classifier_mnli = pipeline("text-classification", model = model_name)
classifier_mnli([
    "She loves me. [SEP] She loves me not. [SEP]",
    "Alice just woke up. [SEP] Alice is awake. [SEP]",
    "I like dogs. [SEP] Everyone likes dogs. [SEP]"])


config.json:   0%|          | 0.00/729 [00:00<?, ?B/s]

c:\Users\mdsha\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mdsha\.cache\huggingface\hub\models--huggingface--distilbert-base-uncased-finetuned-mnli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[{'label': 'contradiction', 'score': 0.9717152714729309},
 {'label': 'entailment', 'score': 0.9119169116020203},
 {'label': 'neutral', 'score': 0.9509281516075134}]